In [1]:
"""
WELLNESS CENTRE ERPNEXT MIGRATION
==================================

Complete data import from 18 CSV files to ERPNext v15.

DATA SCOPE:
- 220 Sales Invoices (KES 2,589,840)
- 220 Payment Entries (all invoices paid)
- 709 Expense transactions (KES 4,360,000)
- 77 Inventory items (KES 944,000)
- 193 Inventory movements
- 45 Contacts (customers, suppliers, employees)
- 25 Events, 54 Room bookings
- Farm operations (eggs, poultry)
- 9 Compliance documents

PHASES:
Phase 1: Master Data + Sales Invoices + Payments
Phase 2: Expenses
Phase 3: Inventory
Phase 4: Operations (Events, Rooms, Farm)

Started: 2026-03-02
"""

print("=" * 70)
print("WELLNESS CENTRE ERPNEXT MIGRATION")
print("=" * 70)
print("Status: STARTING FRESH")
print("Notebook created: 2026-03-02")
print("=" * 70)

WELLNESS CENTRE ERPNEXT MIGRATION
Status: STARTING FRESH
Notebook created: 2026-03-02


In [2]:
# Standard library
from pathlib import Path
from datetime import datetime
import csv
import sys

# Third party
import pandas as pd
from frappeclient import FrappeClient

# Add src to path for imports
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# Paths
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')  # CSV files
OUTPUTS_DIR = REPO_ROOT / 'user-data' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Repo root: {REPO_ROOT}")
print(f"✓ Source: {SRC_DIR}")
print(f"✓ Data: {DATA_DIR}")
print(f"✓ Outputs: {OUTPUTS_DIR}")

✓ Repo root: /home/jovyan/work/ERP/emt
✓ Source: /home/jovyan/work/ERP/emt/src
✓ Data: /home/jovyan/work/ERP/emt/data/wellness_centre
✓ Outputs: /home/jovyan/work/ERP/emt/user-data/outputs


In [3]:
# Load connection config
import yaml
import csv
from pathlib import Path

CONFIG_FILE = REPO_ROOT / 'config' / 'erpnext_connection.yaml'

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Config not found: {CONFIG_FILE}\n"
        f"Copy config/erpnext_connection.yaml.example and fill in credentials"
    )

with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

ERPNEXT_URL = config['url']

# Option 1: Try CSV file first (if specified)
if 'api_csv' in config and config['api_csv']:
    csv_path = REPO_ROOT / config['api_csv']
    
    if csv_path.exists():
        print(f"✓ Loading API keys from CSV: {csv_path.name}")
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            row = next(reader)
            API_KEY = row['api_key']
            API_SECRET = row['api_secret']
        print("  Source: CSV file")
    else:
        print(f"⚠ CSV file not found: {csv_path}")
        print("  Falling back to YAML credentials...")
        API_KEY = config['api_key']
        API_SECRET = config['api_secret']
        print("  Source: YAML file")
else:
    # Option 2: Use YAML credentials
    API_KEY = config['api_key']
    API_SECRET = config['api_secret']
    print("  Source: YAML file")

print(f"✓ Config loaded")
print(f"  URL: {ERPNEXT_URL}")

✓ Loading API keys from CSV: api_keys.csv
  Source: CSV file
✓ Config loaded
  URL: http://erpnext-frontend:8080


In [4]:
# Connect to ERPNext
client = FrappeClient(ERPNEXT_URL)
client.authenticate(API_KEY, API_SECRET)

# Add Host header if using internal Docker URL (for DNS multitenant)
if 'erpnext-frontend' in ERPNEXT_URL or 'localhost' in ERPNEXT_URL:
    # Get domain from config or default
    domain = config.get('headers', {}).get('Host', 'well.rosslyn.cloud')
    client.session.headers.update({"Host": domain})
    print(f"✓ Added Host header: {domain}")

# Test connection
try:
    company = client.get_doc("Company", "Wellness Centre")
    print("=" * 70)
    print("ERPNEXT CONNECTION SUCCESSFUL")
    print("=" * 70)
    print(f"Company: {company.get('company_name')}")
    print(f"Currency: {company.get('default_currency')}")
    print(f"Country: {company.get('country')}")
    print(f"Fiscal Year: {company.get('default_fiscal_year', 'NOT SET')}")
    print("=" * 70)
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

✓ Added Host header: well.rosslyn.cloud
ERPNEXT CONNECTION SUCCESSFUL
Company: Wellness Centre
Currency: KES
Country: Kenya
Fiscal Year: NOT SET


In [5]:
# After connecting to fresh instance, create custom field FIRST
custom_field = {
    "doctype": "Custom Field",
    "dt": "Sales Invoice",
    "fieldname": "original_invoice_number",
    "label": "Original Invoice Number",
    "fieldtype": "Data",
    "insert_after": "naming_series",
    "read_only": 1,
    "allow_on_submit": 1,
    "in_list_view": 1,
    "in_standard_filter": 1
}

try:
    result = client.insert(custom_field)
    print(f"✓ Created custom field: {result['name']}")
except Exception as e:
    if "already exists" in str(e).lower():
        print("✓ Custom field already exists")
    else:
        raise

✓ Custom field already exists


In [6]:
# Load all source CSV files
print("LOADING SOURCE DATA")
print("=" * 70)

# Transactions and categories
transactions_df = pd.read_csv(DATA_DIR / 'transactions.csv')
categories_df = pd.read_csv(DATA_DIR / 'transaction_categories.csv')

# Merge for category names
tx = transactions_df.merge(
    categories_df,
    left_on='category_id',
    right_on='id',
    suffixes=('', '_cat')
)

# Invoice data
invoices_df = pd.read_csv(DATA_DIR / 'etims_invoices.csv')
items_df = pd.read_csv(DATA_DIR / 'etims_invoice_items.csv')

# Contact data
contacts_df = pd.read_csv(DATA_DIR / 'contacts.csv')
contact_types_df = pd.read_csv(DATA_DIR / 'contact_types.csv')

print(f"✓ Transactions:     {len(transactions_df):,}")
print(f"✓ Invoices:         {len(invoices_df):,}")
print(f"✓ Invoice Items:    {len(items_df):,}")
print(f"✓ Contacts:         {len(contacts_df):,}")
print("=" * 70)

LOADING SOURCE DATA
✓ Transactions:     947
✓ Invoices:         220
✓ Invoice Items:    220
✓ Contacts:         45


In [7]:
tx.describe()

,id,category_id,contact_id,property_id,event_id,egg_sale_id,inventory_movement_id,amount,etims_invoice_id,id_cat
count,947.000000,947.000000,836.000000,324.000000,280.000000,103.000000,40.000000,9.470000e+02,220.000000,947.000000
mean,474.000000,11.670539,17.665072,3.432099,13.289286,52.000000,140.775000,1.180815e+04,110.500000,11.670539
std,273.519652,8.388632,14.729425,1.719949,7.025700,29.877528,32.924535,8.362160e+04,63.652704,8.388632
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,85.000000,3.000000e+02,1.000000,1.000000
25%,237.500000,6.000000,6.000000,1.000000,7.000000,26.500000,110.250000,1.500000e+03,55.750000,6.000000
50%,474.000000,7.000000,12.000000,4.000000,14.000000,52.000000,146.000000,1.500000e+03,110.500000,7.000000
75%,710.500000,18.000000,34.000000,5.000000,19.250000,77.500000,169.250000,8.000000e+03,165.250000,18.000000
max,947.000000,31.000000,45.000000,5.000000,25.000000,103.000000,191.000000,2.000000e+06,220.000000,31.000000


## CRITICAL PREREQUISITE: Fiscal Year Configuration

**ERPNext requires fiscal years to exist BEFORE posting transactions.**

### Steps:
1. Search "Fiscal Year" in ERPNext
2. Create fiscal year(s) covering your transaction date range
3. **Assign company to each fiscal year** (Company field)
4. Set one as default (typically current/most recent year)

### For this migration:
- Transaction range: 2024-01-05 to 2025-02-11
- Required: FY-2024 (Jan-Dec 2024) + FY-2025 (Jan-Dec 2025)
- Both assigned to: Wellness Centre

**Without this, all imports will fail with "No Fiscal Year found" errors.**

In [8]:
# Check and auto-fix prerequisites
from setup.prerequisites_checker import PrerequisitesChecker

checker = PrerequisitesChecker(
    client, 
    company="Wellness Centre", 
    data_dir=DATA_DIR
)

# No transactions_df parameter needed - scans CSV files directly
status = checker.check_and_fix_all()

print(status['report'])

if not status['ready']:
    raise SystemExit("❌ CRITICAL: Fix issues above before continuing")

PREREQUISITES CHECK

COMPREHENSIVE DATE SCAN:
  Overall range: 2024-01-05 to 2025-02-28
  Files scanned: 13

  Date ranges by file:
    transactions.csv               transaction_date    : 2024-01-05 to 2025-02-11 (947 records)
    etims_invoices.csv             invoice_date        : 2024-03-01 to 2025-02-11 (220 records)
    etims_invoices.csv             transmission_date   : 2024-03-01 to 2025-02-11 (220 records)
    room_bookings.csv              check_in_date       : 2024-03-16 to 2025-02-11 (54 records)
    room_bookings.csv              check_out_date      : 2024-03-17 to 2025-02-13 (54 records)
    events.csv                     event_date          : 2024-03-16 to 2025-01-11 (25 records)
    events.csv                     end_date            : 2024-03-16 to 2025-01-12 (25 records)
    egg_sales.csv                  sale_date           : 2024-03-01 to 2025-02-07 (103 records)
    egg_production.csv             week_start_date     : 2024-02-12 to 2025-02-03 (52 records)
    egg_p

In [9]:
# Auto-create all master data
from setup.master_data_creator import MasterDataCreator

creator = MasterDataCreator(client, "Wellness Centre")
results = creator.create_all(
    contacts_df=contacts_df,
    contact_types_df=contact_types_df,
    invoices_df=invoices_df,
    invoice_items_df=items_df
)

creator.print_summary()

CREATING MASTER DATA
Creating UOMs...
  Created: 0, Existed: 4, Errors: 0
Creating Customers...
  Created: 0, Existed: 13, Errors: 0
Creating Suppliers...
  Created: 0, Existed: 9, Errors: 0
Creating Service Items...
  Created: 0, Existed: 10, Errors: 0

MASTER DATA CREATION SUMMARY

UOMS:
  Created: 0
  Existed: 4
  Errors:  0

CUSTOMERS:
  Created: 0
  Existed: 13
  Errors:  0

SUPPLIERS:
  Created: 0
  Existed: 9
  Errors:  0

ITEMS:
  Created: 0
  Existed: 10
  Errors:  0


In [10]:
# Check what already exists in ERPNext
print("CURRENT ERPNEXT STATE")
print("=" * 70)

checks = [
    ("Customer", {}, "Customers"),
    ("Supplier", {}, "Suppliers"),
    ("Item", {}, "Items (all)"),
    ("UOM", {}, "Units of Measure"),
    ("Sales Invoice", {"docstatus": 1}, "Sales Invoices (submitted)"),
    ("Sales Invoice", {"docstatus": 0}, "Sales Invoices (draft)"),
    ("Payment Entry", {"docstatus": 1}, "Payment Entries (submitted)"),
    ("Journal Entry", {"docstatus": 1}, "Journal Entries (submitted)"),
]

state = {}
for doctype, filters, label in checks:
    try:
        records = client.get_list(
            doctype, 
            filters=filters, 
            limit_page_length=500
        )
        count = len(records)
        state[label] = count
        print(f"{label:40s}: {count:4d}")
    except Exception as e:
        state[label] = f"ERROR: {str(e)[:30]}"
        print(f"{label:40s}: ERROR - {str(e)[:50]}")

print("=" * 70)

CURRENT ERPNEXT STATE
Customers                               :   13
Suppliers                               :    9
Items (all)                             :   10
Units of Measure                        :  243
Sales Invoices (submitted)              :  220
Sales Invoices (draft)                  :    0
Payment Entries (submitted)             :    1
Journal Entries (submitted)             :    0


In [11]:
print("LOADING SOURCE DATA")
print("=" * 70)

# Analyze by type
print("Transaction Breakdown:")
print(tx.groupby('type').agg({
    'id': 'count',
    'amount': 'sum'
}).rename(columns={'id': 'count', 'amount': 'total'}))

print()
print("=" * 70)

LOADING SOURCE DATA
Transaction Breakdown:
                   count    total
type                             
capital_injection      3  4000000
expense              709  4363477
income               220  2589840
savings               15   229000



In [12]:
# PHASE 1A: Import Sales Invoices
print("=" * 70)
print("PHASE 1A: IMPORTING SALES INVOICES")
print("=" * 70)

from orchestration.sales_invoice_importer import SalesInvoiceImporter

# Use v3.0 sequential importer (proven reliable, optimal for API imports)
importer = SalesInvoiceImporter(
    client,
    "Wellness Centre"
)

# Import batch with timing metrics
results = importer.import_batch(
    invoices_df=invoices_df,
    invoice_items_df=items_df,
    contacts_df=contacts_df  # Required by v3.0 signature (even if unused internally)
)

# Display summary with performance metrics
print(importer.get_summary())

PHASE 1A: IMPORTING SALES INVOICES
[SalesInvoiceImporter 3.0-duplicate-prevention]
Importing 220 sales invoices...
  Skipped 50 duplicates...
  Skipped 100 duplicates...
  Skipped 150 duplicates...
  Skipped 200 duplicates...
SALES INVOICE IMPORT SUMMARY
Successful: 0
Skipped:    220 (already exist)
Failed:     0

Performance:
  Duration: 36.65 seconds (0.61 minutes)
  Rate: 0.0 invoices/second


In [13]:
# CORRECTED VERIFICATION
print("=" * 70)
print("DATA INTEGRITY VERIFICATION (CORRECTED)")
print("=" * 70)

# Get ALL invoices with proper limit
all_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "docstatus", "posting_date", "grand_total"],
    limit_page_length=999  # High limit to get all
)

submitted = [inv for inv in all_invoices if inv['docstatus'] == 1]
draft = [inv for inv in all_invoices if inv['docstatus'] == 0]

print(f"\nTotal invoices retrieved: {len(all_invoices)}")
print(f"  Submitted (docstatus=1): {len(submitted)}")
print(f"  Draft (docstatus=0): {len(draft)}")
print(f"Expected: 220 submitted")

# Verify total amount
if submitted:
    erpnext_total = sum(inv['grand_total'] for inv in submitted)
    source_total = invoices_df['total_amount'].sum()
    
    print(f"\nTotal amounts:")
    print(f"  ERPNext: KES {erpnext_total:,.0f}")
    print(f"  Source:  KES {source_total:,.0f}")
    print(f"  Difference: KES {abs(erpnext_total - source_total):,.0f}")
    print(f"  Match: {'✓' if abs(erpnext_total - source_total) < 1 else '✗'}")

print("=" * 70)

DATA INTEGRITY VERIFICATION (CORRECTED)

Total invoices retrieved: 220
  Submitted (docstatus=1): 220
  Draft (docstatus=0): 0
Expected: 220 submitted

Total amounts:
  ERPNext: KES 2,589,840
  Source:  KES 2,589,840
  Difference: KES 0
  Match: ✓


In [15]:
# PHASE 1B: Import Payment Entries (v3.1)
print("=" * 70)
print("PHASE 1B: IMPORTING PAYMENT ENTRIES")
print("=" * 70)

from orchestration.payment_entry_importer import PaymentEntryImporter

importer = PaymentEntryImporter(client, "Wellness Centre")
results = importer.import_batch(
    transactions_df=transactions_df,
    invoices_df=invoices_df
)

print(importer.get_summary())

PHASE 1B: IMPORTING PAYMENT ENTRIES
[PaymentEntryImporter 3.1-kenyan-accounts-fix]
Importing 220 payment entries...
  Imported 50...
  Imported 100...
  Imported 150...
  Imported 200...
PAYMENT ENTRY IMPORT SUMMARY
Successful: 219
Skipped:    1 (already paid)
Failed:     0

Performance:
  Duration: 822.78 seconds (13.71 minutes)
  Rate: 0.27 payments/second


In [23]:
# Check invoice status
invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "docstatus", "customer", "grand_total", "outstanding_amount"],
    limit_page_length=10
)

print(f"Total invoices found: {len(invoices)}")
print("\nFirst 10 invoices:")
for inv in invoices:
    status = {0: "Draft", 1: "Submitted", 2: "Cancelled"}.get(inv['docstatus'], "Unknown")
    print(f"  {inv['name']}: {status}, Outstanding: {inv['outstanding_amount']}")

# Count by status
from collections import Counter
status_counts = Counter([inv['docstatus'] for inv in invoices])
print(f"\nStatus breakdown:")
for docstatus, count in status_counts.items():
    status_name = {0: "Draft", 1: "Submitted", 2: "Cancelled"}.get(docstatus, "Unknown")
    print(f"  {status_name}: {count}")

Total invoices found: 10

First 10 invoices:
  ACC-SINV-2026-00001: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00002: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00003: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00004: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00005: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00006: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00007: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00008: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00009: Submitted, Outstanding: 0.0
  ACC-SINV-2026-00010: Submitted, Outstanding: 0.0

Status breakdown:
  Submitted: 10


In [24]:
# Get ALL invoices (increase limit)
all_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "docstatus", "posting_date", "grand_total", "outstanding_amount"],
    limit_page_length=999  # Get all
)

print(f"Total invoices: {len(all_invoices)}")

# Count by status
from collections import Counter
status_counts = Counter([inv['docstatus'] for inv in all_invoices])
print(f"\nStatus breakdown:")
for docstatus, count in status_counts.items():
    status_name = {0: "Draft", 1: "Submitted", 2: "Cancelled"}.get(docstatus, "Unknown")
    print(f"  {status_name}: {count}")

# Calculate totals
submitted = [inv for inv in all_invoices if inv['docstatus'] == 1]
total_amount = sum(inv['grand_total'] for inv in submitted)
total_outstanding = sum(inv['outstanding_amount'] for inv in submitted)

print(f"\nSubmitted invoices:")
print(f"  Count: {len(submitted)}")
print(f"  Total Amount: KES {total_amount:,.2f}")
print(f"  Outstanding: KES {total_outstanding:,.2f}")

Total invoices: 220

Status breakdown:
  Submitted: 220

Submitted invoices:
  Count: 220
  Total Amount: KES 2,589,840.00
  Outstanding: KES 0.00


In [25]:
# Check the Sales Invoice meta to see all fields
invoice_meta = client.get_doc("DocType", "Sales Invoice")

# Look for fields with "submit" in the name
submit_fields = [
    field for field in invoice_meta.get('fields', [])
    if 'submit' in field.get('fieldname', '').lower() or 
       'submit' in field.get('label', '').lower()
]

print("Fields related to 'submit':")
for field in submit_fields:
    print(f"  {field.get('fieldname')}: {field.get('label')} ({field.get('fieldtype')})")
    if field.get('options'):
        print(f"    Options: {field.get('options')}")

# Also check one actual invoice to see what this field contains
sample = client.get_doc("Sales Invoice", "ACC-SINV-2026-00001")
print("\nSample invoice docstatus-related fields:")
print(f"  docstatus: {sample.get('docstatus')}")
print(f"  status: {sample.get('status')}")
for key, value in sample.items():
    if 'submit' in str(key).lower() and value:
        print(f"  {key}: {value}")

Fields related to 'submit':

Sample invoice docstatus-related fields:
  docstatus: 1
  status: Paid


In [22]:
# Check what other payment methods exist
payment_methods = transactions_df['payment_method'].unique()
print("Payment methods in data:")
for method in payment_methods:
    count = len(transactions_df[transactions_df['payment_method'] == method])
    print(f"  {method}: {count} transactions")

Payment methods in data:
  Bank Transfer: 138 transactions
  M-Pesa: 513 transactions
  Cash: 296 transactions


In [26]:
print("PHASE 2 SCOPE - WHAT NEEDS IMPORTING")
print("=" * 70)

# Expenses (709 records)
expenses = tx[tx['type'] == 'expense']
print(f"Expenses:           {len(expenses):4d} records, KES {expenses['amount'].sum():,.0f}")

# Capital injections (3 records) 
capital = tx[tx['type'] == 'capital_injection']
print(f"Capital Injection:  {len(capital):4d} records, KES {capital['amount'].sum():,.0f}")

# Savings (15 records)
savings = tx[tx['type'] == 'savings']
print(f"Savings:            {len(savings):4d} records, KES {savings['amount'].sum():,.0f}")

print()
print(f"TOTAL TO IMPORT:    {len(expenses) + len(capital) + len(savings):4d} records")
print("=" * 70)

# Expense breakdown by category (using 'name' from categories table)
print("\nExpense Categories (Top 15 by total amount):")
expense_by_cat = expenses.groupby('name')['amount'].agg(['count', 'sum']).sort_values('sum', ascending=False)
print(expense_by_cat.head(15).to_string())
print()

# Payment method breakdown for expenses
print("\nExpense Payment Methods:")
payment_methods = expenses.groupby('payment_method')['amount'].agg(['count', 'sum'])
print(payment_methods.to_string())

print("=" * 70)

PHASE 2 SCOPE - WHAT NEEDS IMPORTING
Expenses:            709 records, KES 4,363,477
Capital Injection:     3 records, KES 4,000,000
Savings:              15 records, KES 229,000

TOTAL TO IMPORT:     727 records

Expense Categories (Top 15 by total amount):
                                         count     sum
name                                                  
Inventory Purchases                         53  809250
Permanent Staff Salaries                    25  576000
Property Renovations                         6  465000
Animal Feed (Chicken)                       23  358627
Garden & Landscaping Maintenance            40  330000
Casual Labour — Security                   167  250500
Event Supplies & Setup                      25  184396
Casual Labour — Cleaning & Housekeeping    119  178500
Estate Service Charge                       14  168000
Supplies & Provisions                       27  105500
Livestock Acquisition                        1  100000
Agent Commissions         

In [27]:
# Check what expense accounts already exist in ERPNext
print("CURRENT CHART OF ACCOUNTS - EXPENSE ACCOUNTS")
print("=" * 70)

accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Expense"
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=100
)

print(f"Total expense accounts: {len(accounts)}\n")

for acc in accounts:
    print(f"  {acc['account_name']:50s} ({acc.get('account_type', 'N/A')})")

print("=" * 70)

CURRENT CHART OF ACCOUNTS - EXPENSE ACCOUNTS
Total expense accounts: 30

  Administrative Expenses                            ()
  Commission on Sales                                ()
  Cost of Goods Sold                                 (Cost of Goods Sold)
  Depreciation                                       (Depreciation)
  Direct Expenses                                    ()
  Entertainment Expenses                             ()
  Exchange Gain/Loss                                 ()
  Expenses                                           ()
  Expenses Included In Asset Valuation               (Expenses Included In Asset Valuation)
  Expenses Included In Valuation                     (Expenses Included In Valuation)
  Freight and Forwarding Charges                     (Chargeable)
  Gain/Loss on Asset Disposal                        ()
  Impairment                                         ()
  Indirect Expenses                                  ()
  Legal Expenses                     

In [28]:
# Install PyYAML for config file parsing
import sys
import subprocess

print("Installing PyYAML...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "pyyaml>=6.0"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ PyYAML installed successfully")
else:
    print(f"Installation output: {result.stdout}")
    print(f"Errors: {result.stderr}")

Installing PyYAML...
✓ PyYAML installed successfully


In [30]:
# Import account mapper
from orchestration.account_mapper import AccountMapper

print("MAPPING EXPENSE CATEGORIES TO ACCOUNTS")
print("=" * 70)

# Initialize mapper with config
config_file = REPO_ROOT / 'config' / 'account_mappings.yaml'
mapper = AccountMapper(config_file=config_file, company="Wellness Centre")

# Map all expense categories
expense_mappings = mapper.map_categories(
    categories_df, 
    transaction_type='expense'
)

print(f"Total categories mapped: {len(expense_mappings)}\n")
print("Mapping Results:")
print(expense_mappings[['category_name', 'erpnext_account', 'create_if_missing']].to_string(index=False))

print("\n" + "=" * 70)
print(f"Accounts to create: {expense_mappings['create_if_missing'].sum()}")
print("=" * 70)

MAPPING EXPENSE CATEGORIES TO ACCOUNTS
Total categories mapped: 24

Mapping Results:
                          category_name                              erpnext_account  create_if_missing
               Permanent Staff Salaries                                  Salary - WC              False
               Casual Labour — Security                Casual Labour — Security - WC               True
Casual Labour — Cleaning & Housekeeping Casual Labour — Cleaning & Housekeeping - WC               True
  Casual Labour — Maintenance & Repairs   Casual Labour — Maintenance & Repairs - WC               True
              Casual Labour — Transport               Casual Labour — Transport - WC               True
            Casual Labour — Event Setup             Casual Labour — Event Setup - WC               True
                  Animal Feed (Chicken)                   Animal Feed (Chicken) - WC               True
                        Veterinary Care                         Veterinary Care - W

In [31]:
# Create accounts that don't exist
print("CREATING MISSING EXPENSE ACCOUNTS")
print("=" * 70)

results = mapper.create_missing_accounts(client, expense_mappings)

print(f"Created:  {len(results['created'])}")
print(f"Existed:  {len(results['existed'])}")
print(f"Errors:   {len(results['errors'])}")
print()

if results['created']:
    print("Newly created accounts:")
    for account in results['created']:
        print(f"  ✓ {account}")
    print()

if results['errors']:
    print("Errors encountered:")
    for err in results['errors']:
        print(f"  ✗ {err['category']}: {err['error']}")
    print()

print("=" * 70)

CREATING MISSING EXPENSE ACCOUNTS
Created:  17
Existed:  0
Errors:   0

Newly created accounts:
  ✓ Casual Labour — Security
  ✓ Casual Labour — Cleaning & Housekeeping
  ✓ Casual Labour — Maintenance & Repairs
  ✓ Casual Labour — Transport
  ✓ Casual Labour — Event Setup
  ✓ Animal Feed (Chicken)
  ✓ Veterinary Care
  ✓ Garden & Landscaping Maintenance
  ✓ Property Renovations
  ✓ Consultant Fees
  ✓ Supplies & Provisions
  ✓ Dog Care
  ✓ Event Supplies & Setup
  ✓ Livestock Acquisition
  ✓ Business Registration & Permits
  ✓ Furniture & Fittings
  ✓ Estate Service Charge



In [32]:
# Check which bank/cash accounts exist for payment mapping
print("PAYMENT ACCOUNTS FOR EXPENSE TRANSACTIONS")
print("=" * 70)

payment_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "account_type": ["in", ["Bank", "Cash"]]
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=50
)

print(f"Total payment accounts: {len(payment_accounts)}\n")

for acc in payment_accounts:
    acc_type = acc.get('account_type', 'N/A')
    print(f"  {acc['account_name']:40s} ({acc_type})")

print("\n" + "=" * 70)

# Show expense payment method breakdown again
print("Expense Payment Methods (reminder):")
print(expenses.groupby('payment_method')['amount'].agg(['count', 'sum']).to_string())
print("=" * 70)

PAYMENT ACCOUNTS FOR EXPENSE TRANSACTIONS
Total payment accounts: 5

  Bank - KCB                               (Bank)
  Bank Accounts                            (Bank)
  M-Pesa                                   (Bank)
  Cash                                     (Cash)
  Cash In Hand                             (Cash)

Expense Payment Methods (reminder):
                count      sum
payment_method                
Bank Transfer      65  1760676
Cash              271   636729
M-Pesa            373  1966072


In [33]:
# Import expense importer
from orchestration.expense_importer import ExpenseImporter

print("TESTING EXPENSE IMPORT (10 records)")
print("=" * 70)

# Initialize importer
expense_imp = ExpenseImporter(
    client=client,
    company="Wellness Centre",
    company_suffix="WC"
)

# Test with first 10 expenses
test_result = expense_imp.import_expenses(
    transactions_df=tx,
    account_mappings=expense_mappings,
    auto_submit=True,
    limit=10
)

# Show results
expense_imp.print_summary()

TESTING EXPENSE IMPORT (10 records)
Importing 10 expense transactions...
  Progress: 10/10 (✓ 6, ✗ 4)

EXPENSE IMPORT SUMMARY
Total attempted:  10
Succeeded:        6
Failed:           4
Success amount:   KES 25,500.00

First 5 failures:
  ID 2: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 3: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 5: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 9: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api

In [34]:
# Get full error details from one failure
if test_result['failures']:
    first_failure = test_result['failures'][0]
    print("Full error for first failure:")
    print(f"Transaction ID: {first_failure['transaction_id']}")
    print(f"Error: {first_failure['error']}")
    
    # Also check which transactions failed
    print("\nFailed transaction details:")
    failed_ids = [f['transaction_id'] for f in test_result['failures']]
    failed_tx = tx[tx['id'].isin(failed_ids)]
    print(failed_tx[['id', 'transaction_date', 'type', 'category_id', 'amount', 'payment_method']].to_string())

Full error for first failure:
Transaction ID: 2
Error: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app

Failed transaction details:
   id transaction_date     type  category_id  amount payment_method
1   2       2024-01-16  expense           24   15000  Bank Transfer
2   3       2024-01-17  expense           24    8000         M-Pesa
4   5       2024-01-18  expense           15   25000         M-Pesa
8   9       2024-01-21  expense           15   35000         M-Pesa


In [36]:
# Check what category_id 24 and 15 are
print("Failed transaction categories:")
failed_cats = tx[tx['id'].isin([2, 3, 5, 9])].merge(
    categories_df,  # Your variable name
    left_on='category_id',
    right_on='id',
    suffixes=('', '_cat')
)
print(failed_cats[['id', 'name', 'amount', 'payment_method']].to_string())

# Try to manually create one journal entry to see the real error
test_tx = tx[tx['id'] == 2].iloc[0]
print(f"\nTrying to manually create Journal Entry for transaction {test_tx['id']}...")

# Get the account mapping
test_mapping = expense_mappings[expense_mappings['category_id'] == test_tx['category_id']].iloc[0]
print(f"Category: {test_mapping['category_name']}")
print(f"Account: {test_mapping['erpnext_account']}")

# Try to build and insert the journal entry
try:
    je_doc = {
        "doctype": "Journal Entry",
        "posting_date": pd.to_datetime(test_tx['transaction_date']).strftime('%Y-%m-%d'),
        "company": "Wellness Centre",
        "accounts": [
            {
                "account": test_mapping['erpnext_account'],
                "debit_in_account_currency": float(test_tx['amount']),
                "credit_in_account_currency": 0
            },
            {
                "account": "Bank - KCB - WC",  # For Bank Transfer
                "debit_in_account_currency": 0,
                "credit_in_account_currency": float(test_tx['amount'])
            }
        ]
    }
    
    created = client.insert(je_doc)
    print(f"✓ Journal Entry created: {created['name']}")
    
except Exception as e:
    print(f"✗ Error:")
    import traceback
    traceback.print_exc()
    print(f"\nFull error: {str(e)}")

Failed transaction categories:
   id                              name  amount payment_method
0   2   Business Registration & Permits   15000  Bank Transfer
1   3   Business Registration & Permits    8000         M-Pesa
2   5  Garden & Landscaping Maintenance   25000         M-Pesa
3   9  Garden & Landscaping Maintenance   35000         M-Pesa

Trying to manually create Journal Entry for transaction 2...
Category: Business Registration & Permits
Account: Business Registration & Permits - WC
✓ Journal Entry created: ACC-JV-2026-00007


In [37]:
# Check the created journal entry
je = client.get_doc("Journal Entry", "ACC-JV-2026-00007")
print("Successfully created Journal Entry:")
print(f"  Posting Date: {je['posting_date']}")
print(f"  Total Debit: {je['total_debit']}")
print(f"  Total Credit: {je['total_credit']}")
print(f"  Accounts:")
for acc in je['accounts']:
    print(f"    {acc['account']}: Debit={acc.get('debit_in_account_currency', 0)}, Credit={acc.get('credit_in_account_currency', 0)}")

# Now check what the ExpenseImporter is actually building
print("\n" + "="*70)
print("Let's see what ExpenseImporter built for transaction 2:")

# Get the importer's method
import inspect
source = inspect.getsource(expense_imp._build_journal_entry)
print(source[:1500])  # Show first part of the method

Successfully created Journal Entry:
  Posting Date: 2024-01-16
  Total Debit: 15000.0
  Total Credit: 15000.0
  Accounts:
    Business Registration & Permits - WC: Debit=15000.0, Credit=0.0
    Bank - KCB - WC: Debit=0.0, Credit=15000.0

Let's see what ExpenseImporter built for transaction 2:


AttributeError: 'ExpenseImporter' object has no attribute '_build_journal_entry'

In [38]:
# Check ExpenseImporter methods
print("ExpenseImporter methods:")
methods = [method for method in dir(expense_imp) if not method.startswith('_') and callable(getattr(expense_imp, method))]
for method in methods:
    print(f"  {method}")

# Also check private methods
print("\nPrivate methods:")
private_methods = [method for method in dir(expense_imp) if method.startswith('_') and callable(getattr(expense_imp, method)) and not method.startswith('__')]
for method in private_methods:
    print(f"  {method}")

# Show the source of the import_expenses method
import inspect
print("\n" + "="*70)
print("ExpenseImporter.import_expenses source code:")
print("="*70)
source = inspect.getsource(expense_imp.import_expenses)
print(source)

ExpenseImporter methods:
  build_journal_entry
  get_summary
  import_expenses
  print_summary

Private methods:
  _get_payment_account

ExpenseImporter.import_expenses source code:
    def import_expenses(
        self,
        transactions_df: pd.DataFrame,
        account_mappings: pd.DataFrame,
        auto_submit: bool = True,
        limit: Optional[int] = None
    ) -> Dict[str, any]:
        """
        Import expense transactions as Journal Entries.
        
        Args:
            transactions_df: DataFrame with expense transactions
            account_mappings: DataFrame from AccountMapper.map_categories()
            auto_submit: Submit journal entries after creation
            limit: Optional limit for testing (e.g., 10 for first batch)
            
        Returns:
            Dictionary with import results
        """
        # Filter to expenses only
        expenses = transactions_df[transactions_df['type'] == 'expense'].copy()
        
        if limit:
           

In [39]:
# Show the build_journal_entry source
import inspect
print("ExpenseImporter.build_journal_entry source code:")
print("="*70)
source = inspect.getsource(expense_imp.build_journal_entry)
print(source)

ExpenseImporter.build_journal_entry source code:
    def build_journal_entry(
        self,
        transaction: dict,
        expense_account: str
    ) -> dict:
        """
        Build Journal Entry payload for an expense transaction.
        
        Creates double-entry:
        - Debit expense account (expense increases)
        - Credit payment account (cash/bank decreases)
        
        Args:
            transaction: Transaction record from CSV
            expense_account: Mapped expense account name
            
        Returns:
            ERPNext Journal Entry payload
        """
        # Get payment account
        payment_method = transaction['payment_method']
        payment_account = self._get_payment_account(payment_method)
        
        # Build journal entry
        amount = float(transaction['amount'])
        
        payload = {
            "doctype": "Journal Entry",
            "company": self.company,
            "posting_date": transaction['transaction_d

In [40]:
# Show _get_payment_account
import inspect
print("ExpenseImporter._get_payment_account source code:")
print("="*70)
source = inspect.getsource(expense_imp._get_payment_account)
print(source)

print("\n" + "="*70)
print("Comparison with working manual entry:")
print("="*70)

# Our working manual entry used
print("Manual entry (WORKED):")
print("  posting_date: '2024-01-16' (string)")
print("  No voucher_type field")
print("  No user_remark field")
print("  No reference_type/reference_name fields")

print("\nExpenseImporter uses:")
print("  posting_date: transaction['transaction_date'] (might be datetime object)")
print("  voucher_type: 'Journal Entry'")
print("  user_remark: transaction.get('description', '')")
print("  reference_type: '' (empty string)")
print("  reference_name: '' (empty string)")

# Check what transaction_date actually is
test_tx = tx[tx['id'] == 2].iloc[0]
print(f"\nTest transaction date type: {type(test_tx['transaction_date'])}")
print(f"Test transaction date value: {test_tx['transaction_date']}")

ExpenseImporter._get_payment_account source code:
    def _get_payment_account(self, payment_method: str) -> str:
        """
        Get ERPNext account for payment method.
        
        Args:
            payment_method: Payment method from transaction (M-Pesa/Bank Transfer/Cash)
            
        Returns:
            ERPNext account name with company suffix
            
        Raises:
            ValueError: If payment method not recognized
        """
        template = self.PAYMENT_ACCOUNT_MAP.get(payment_method)
        if not template:
            raise ValueError(f"Unknown payment method: {payment_method}")
        
        return template.format(suffix=self.suffix)


Comparison with working manual entry:
Manual entry (WORKED):
  posting_date: '2024-01-16' (string)
  No voucher_type field
  No user_remark field
  No reference_type/reference_name fields

ExpenseImporter uses:
  posting_date: transaction['transaction_date'] (might be datetime object)
  voucher_type: 'Journa

In [41]:
# Check the payment account mapping
print("ExpenseImporter.PAYMENT_ACCOUNT_MAP:")
print(expense_imp.PAYMENT_ACCOUNT_MAP)

print("\nCompany suffix:")
print(f"  {expense_imp.suffix}")

# Test what account it generates for Bank Transfer
test_account = expense_imp._get_payment_account("Bank Transfer")
print(f"\nBank Transfer maps to: {test_account}")

# Check if this account exists
try:
    acc_check = client.get_list(
        "Account",
        filters={"name": test_account},
        fields=["name", "account_type"],
        limit_page_length=1
    )
    if acc_check:
        print(f"✓ Account exists: {acc_check[0]}")
    else:
        print(f"✗ Account NOT found: {test_account}")
except Exception as e:
    print(f"✗ Error checking account: {e}")

# Also try building the full journal entry for transaction 2
print("\n" + "="*70)
print("Building journal entry with ExpenseImporter.build_journal_entry:")
test_tx = tx[tx['id'] == 2].iloc[0]
test_mapping = expense_mappings[expense_mappings['category_id'] == test_tx['category_id']].iloc[0]

payload = expense_imp.build_journal_entry(
    test_tx.to_dict(),
    test_mapping['erpnext_account']
)

print("Generated payload:")
import json
print(json.dumps(payload, indent=2, default=str))

ExpenseImporter.PAYMENT_ACCOUNT_MAP:
{'M-Pesa': 'Mobile Money - {suffix}', 'Bank Transfer': 'KCB - {suffix}', 'Cash': 'Cash - {suffix}'}

Company suffix:
  WC

Bank Transfer maps to: KCB - WC
✗ Account NOT found: KCB - WC

Building journal entry with ExpenseImporter.build_journal_entry:
Generated payload:
{
  "doctype": "Journal Entry",
  "company": "Wellness Centre",
  "posting_date": "2024-01-16",
  "voucher_type": "Journal Entry",
  "user_remark": "Business registration \u2014 county government",
  "accounts": [
    {
      "account": "Business Registration & Permits - WC",
      "debit_in_account_currency": 15000.0,
      "credit_in_account_currency": 0,
      "reference_type": "",
      "reference_name": ""
    },
    {
      "account": "KCB - WC",
      "debit_in_account_currency": 0,
      "credit_in_account_currency": 15000.0,
      "reference_type": "",
      "reference_name": ""
    }
  ]
}


In [42]:
# The ExpenseImporter should discover accounts, not hard-code them
# Let's see what payment accounts actually exist

payment_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "account_type": ["in", ["Bank", "Cash"]],
        "is_group": 0  # Only leaf accounts
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=50
)

print("Available payment accounts:")
for acc in payment_accounts:
    print(f"  {acc['name']} (type: {acc['account_type']})")

# Build dynamic mapping based on what exists
print("\nSuggested dynamic mapping logic:")
print("1. M-Pesa → Search for account with 'M-Pesa' or 'Mobile Money' in name")
print("2. Bank Transfer → Search for Bank type accounts (or specific bank name)")
print("3. Cash → Search for Cash type accounts")

# Or better: Let the AccountMapper handle payment accounts too
print("\nBest solution: Extend AccountMapper to handle payment accounts")
print("Add payment_account_mappings.yaml with:")
print("  M-Pesa: M-Pesa - WC")
print("  Bank Transfer: Bank - KCB - WC")
print("  Cash: Cash - WC")

Available payment accounts:
  Bank - KCB - WC (type: Bank)
  M-Pesa - WC (type: Bank)
  Cash - WC (type: Cash)

Suggested dynamic mapping logic:
1. M-Pesa → Search for account with 'M-Pesa' or 'Mobile Money' in name
2. Bank Transfer → Search for Bank type accounts (or specific bank name)
3. Cash → Search for Cash type accounts

Best solution: Extend AccountMapper to handle payment accounts
Add payment_account_mappings.yaml with:
  M-Pesa: M-Pesa - WC
  Bank Transfer: Bank - KCB - WC
  Cash: Cash - WC


In [43]:
# Check if the ExpenseImporter was modified or reloaded
print("Current ExpenseImporter PAYMENT_ACCOUNT_MAP:")
print(expense_imp.PAYMENT_ACCOUNT_MAP)

# Compare to what we saw earlier
print("\nWhat we saw before (broken):")
print("{'M-Pesa': 'Mobile Money - {suffix}', 'Bank Transfer': 'KCB - {suffix}', 'Cash': 'Cash - {suffix}'}")

# Check if you reloaded the module
import sys
if 'orchestration.expense_importer' in sys.modules:
    print("\nExpenseImporter module info:")
    print(f"  Module: {sys.modules['orchestration.expense_importer']}")

Current ExpenseImporter PAYMENT_ACCOUNT_MAP:
{'M-Pesa': 'Mobile Money - {suffix}', 'Bank Transfer': 'KCB - {suffix}', 'Cash': 'Cash - {suffix}'}

What we saw before (broken):
{'M-Pesa': 'Mobile Money - {suffix}', 'Bank Transfer': 'KCB - {suffix}', 'Cash': 'Cash - {suffix}'}

ExpenseImporter module info:
  Module: <module 'orchestration.expense_importer' from '/home/jovyan/work/ERP/emt/src/orchestration/expense_importer.py'>


In [44]:
# Check the journal entries that were created
recent_je = client.get_list(
    "Journal Entry",
    filters={"docstatus": 1},
    fields=["name", "posting_date", "total_debit", "total_credit"],
    order_by="creation desc",
    limit_page_length=10
)

print(f"Recent Journal Entries created: {len(recent_je)}")
print("\nFirst 5:")
for je in recent_je[:5]:
    print(f"  {je['name']}: {je['posting_date']}, Debit={je['total_debit']}")

# Check one journal entry's accounts in detail
if recent_je:
    sample_je = client.get_doc("Journal Entry", recent_je[0]['name'])
    print(f"\nSample Journal Entry: {sample_je['name']}")
    print("Accounts:")
    for acc in sample_je['accounts']:
        print(f"  {acc['account']}: Debit={acc.get('debit_in_account_currency', 0)}, Credit={acc.get('credit_in_account_currency', 0)}")

Recent Journal Entries created: 6

First 5:
  ACC-JV-2026-00006: 2024-01-22, Debit=1500.0
  ACC-JV-2026-00005: 2024-01-22, Debit=1500.0
  ACC-JV-2026-00004: 2024-01-20, Debit=1500.0
  ACC-JV-2026-00003: 2024-01-20, Debit=18000.0
  ACC-JV-2026-00002: 2024-01-19, Debit=1500.0

Sample Journal Entry: ACC-JV-2026-00006
Accounts:
  Casual Labour — Maintenance & Repairs - WC: Debit=1500.0, Credit=0.0
  Cash - WC: Debit=0.0, Credit=1500.0


In [45]:
# Check ALL bank/cash accounts
all_payment_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "account_type": ["in", ["Bank", "Cash"]]
    },
    fields=["name", "account_name", "is_group"],
    limit_page_length=50
)

print("ALL Bank/Cash accounts (including groups):")
for acc in all_payment_accounts:
    group_status = "GROUP" if acc.get('is_group') else "LEAF"
    print(f"  {acc['name']:40s} [{group_status}]")

print("\n" + "="*70)
print("ExpenseImporter is looking for:")
print("  'Mobile Money - WC' (doesn't exist)")
print("  'KCB - WC' (doesn't exist)")  
print("  'Cash - WC' (exists)")

print("\nWhat likely existed in the PREVIOUS session:")
print("  Maybe: 'Mobile Money - WC' account existed then?")
print("  Maybe: 'KCB - WC' account existed then?")

ALL Bank/Cash accounts (including groups):
  Bank - KCB - WC                          [LEAF]
  Bank Accounts - WC                       [GROUP]
  M-Pesa - WC                              [LEAF]
  Cash - WC                                [LEAF]
  Cash In Hand - WC                        [GROUP]

ExpenseImporter is looking for:
  'Mobile Money - WC' (doesn't exist)
  'KCB - WC' (doesn't exist)
  'Cash - WC' (exists)

What likely existed in the PREVIOUS session:
  Maybe: 'Mobile Money - WC' account existed then?
  Maybe: 'KCB - WC' account existed then?


In [46]:
# Check if any transactions reference the new accounts
payments_using_mpesa = client.get_list(
    "Payment Entry",
    filters={
        "paid_to": "M-Pesa - WC",
        "docstatus": 1
    },
    fields=["name"],
    limit_page_length=5
)

payments_using_bank = client.get_list(
    "Payment Entry",
    filters={
        "paid_to": "Bank - KCB - WC",
        "docstatus": 1
    },
    fields=["name"],
    limit_page_length=5
)

print(f"Payment Entries using M-Pesa - WC: {len(payments_using_mpesa)}")
print(f"Payment Entries using Bank - KCB - WC: {len(payments_using_bank)}")

if payments_using_mpesa or payments_using_bank:
    print("\n⚠ WARNING: Accounts are already in use by submitted documents")
    print("Renaming will be complex - ERPNext may not allow it")
    print("\nBetter option: Update ExpenseImporter code instead")
else:
    print("\n✓ Accounts not yet used - safe to rename")

Payment Entries using M-Pesa - WC: 5
Payment Entries using Bank - KCB - WC: 5

⚠ WARNING: Accounts are already in use by submitted documents
Renaming will be complex - ERPNext may not allow it

Better option: Update ExpenseImporter code instead


In [25]:
# Import all 709 expense transactions
print("IMPORTING ALL EXPENSE TRANSACTIONS")
print("=" * 70)

# Create fresh importer instance (clears success/failure lists)
expense_imp = ExpenseImporter(
    client=client,
    company="Wellness Centre",
    company_suffix="WC"
)

# Import all expenses (no limit)
result = expense_imp.import_expenses(
    transactions_df=tx,
    account_mappings=expense_mappings,
    auto_submit=True,
    limit=None  # Import all
)

# Show results
expense_imp.print_summary()

# Verify total matches source data
expected_total = expenses['amount'].sum()
actual_total = result['success_amount']
difference = abs(expected_total - actual_total)

print(f"\nVERIFICATION:")
print(f"  Expected total: KES {expected_total:,.2f}")
print(f"  Imported total: KES {actual_total:,.2f}")
print(f"  Difference:     KES {difference:,.2f}")

if difference < 100 and result['failed'] == 0:
    print("\n✓ ALL EXPENSES IMPORTED SUCCESSFULLY")
else:
    print("\n⚠ Review failures above")

IMPORTING ALL EXPENSE TRANSACTIONS
Importing 709 expense transactions...
  Progress: 50/709 (✓ 50, ✗ 0)
  Progress: 100/709 (✓ 100, ✗ 0)
  Progress: 150/709 (✓ 150, ✗ 0)
  Progress: 200/709 (✓ 200, ✗ 0)
  Progress: 250/709 (✓ 250, ✗ 0)
  Progress: 300/709 (✓ 300, ✗ 0)
  Progress: 350/709 (✓ 350, ✗ 0)
  Progress: 400/709 (✓ 400, ✗ 0)
  Progress: 450/709 (✓ 450, ✗ 0)
  Progress: 500/709 (✓ 500, ✗ 0)
  Progress: 550/709 (✓ 550, ✗ 0)
  Progress: 600/709 (✓ 600, ✗ 0)
  Progress: 650/709 (✓ 650, ✗ 0)
  Progress: 700/709 (✓ 700, ✗ 0)
  Progress: 709/709 (✓ 709, ✗ 0)

EXPENSE IMPORT SUMMARY
Total attempted:  709
Succeeded:        709
Failed:           0
Success amount:   KES 4,363,477.00

VERIFICATION:
  Expected total: KES 4,363,477.00
  Imported total: KES 4,363,477.00
  Difference:     KES 0.00

✓ ALL EXPENSES IMPORTED SUCCESSFULLY


In [26]:
# Examine capital and savings transactions
print("CAPITAL & SAVINGS TRANSACTIONS")
print("=" * 70)

# Capital injections
capital_tx = tx[tx['type'] == 'capital_injection']
print(f"\nCapital Injections: {len(capital_tx)} transactions")
print(capital_tx[['transaction_date', 'name', 'amount', 'payment_method', 'description']].to_string(index=False))

print("\n" + "=" * 70)

# Savings
savings_tx = tx[tx['type'] == 'savings']
print(f"\nSavings: {len(savings_tx)} transactions")
print(savings_tx[['transaction_date', 'name', 'amount', 'payment_method', 'description']].head(10).to_string(index=False))

print("\n" + "=" * 70)
print(f"Capital total:  KES {capital_tx['amount'].sum():,.0f}")
print(f"Savings total:  KES {savings_tx['amount'].sum():,.0f}")
print("=" * 70)

CAPITAL & SAVINGS TRANSACTIONS

Capital Injections: 3 transactions
transaction_date                    name  amount payment_method                                           description
      2024-01-15 Owner Capital Injection 2000000  Bank Transfer Owner capital injection — initial business setup fund
      2024-01-22 Owner Capital Injection 1500000  Bank Transfer Owner capital injection — property prep and inventory
      2024-02-20 Owner Capital Injection  500000  Bank Transfer     Owner capital injection — renovation cost overrun


Savings: 15 transactions
transaction_date                             name  amount payment_method                                          description
      2024-04-26         Savings — Generator Fund   15000  Bank Transfer         Savings — Generator Fund set-aside (2024-04)
      2024-05-28         Savings — Generator Fund   10000  Bank Transfer         Savings — Generator Fund set-aside (2024-05)
      2024-06-27 Savings — Large Events Tent Fund   1200

In [27]:
# Check if equity and savings accounts exist
print("CHECKING REQUIRED ACCOUNTS")
print("=" * 70)

# Check for Owner Capital account (equity)
equity_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Equity"
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=20
)

print("Equity Accounts:")
for acc in equity_accounts:
    print(f"  {acc['account_name']}")

print()

# Check for Savings/Bank accounts (assets)
asset_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Asset",
        "account_type": "Bank"
    },
    fields=["name", "account_name"],
    limit_page_length=20
)

print("Bank/Asset Accounts:")
for acc in asset_accounts:
    print(f"  {acc['account_name']}")

print()
print("=" * 70)

# Strategy decision
print("\nIMPORT STRATEGY:")
print("1. Capital Injections → Journal Entry")
print("   Debit: KCB (bank increases)")
print("   Credit: Owner Capital/Equity (owner equity increases)")
print()
print("2. Savings Transfers → Journal Entry")
print("   Debit: Savings Account (savings increases)")
print("   Credit: KCB (operating bank decreases)")
print("=" * 70)

CHECKING REQUIRED ACCOUNTS
Equity Accounts:
  Capital Stock
  Dividends Paid
  Equity
  Opening Balance Equity
  Retained Earnings
  Revaluation Surplus

Bank/Asset Accounts:
  Bank Accounts
  KCB
  Mobile Money


IMPORT STRATEGY:
1. Capital Injections → Journal Entry
   Debit: KCB (bank increases)
   Credit: Owner Capital/Equity (owner equity increases)

2. Savings Transfers → Journal Entry
   Debit: Savings Account (savings increases)
   Credit: KCB (operating bank decreases)


In [28]:
# Create Savings Account under Bank Accounts
print("CREATING SAVINGS ACCOUNT")
print("=" * 70)

try:
    # Check if already exists
    existing = client.get_list(
        "Account",
        filters={"name": "Savings Account - WC"},
        limit_page_length=1
    )
    
    if existing:
        print("✓ Savings Account already exists")
    else:
        # Create new bank account
        payload = {
            "doctype": "Account",
            "account_name": "Savings Account",
            "parent_account": "Bank Accounts - WC",
            "company": "Wellness Centre",
            "is_group": 0,
            "account_type": "Bank"
        }
        
        result = client.insert(payload)
        print("✓ Created: Savings Account - WC")
        
except Exception as e:
    print(f"✗ Error: {str(e)[:200]}")

print("=" * 70)

CREATING SAVINGS ACCOUNT
✓ Created: Savings Account - WC


In [29]:
# Import capital injections using journal entries
print("IMPORTING CAPITAL INJECTIONS")
print("=" * 70)

capital_count = 0
capital_errors = []

for _, tx_row in capital_tx.iterrows():
    try:
        amount = float(tx_row['amount'])
        
        # Journal Entry: Debit Bank, Credit Equity
        payload = {
            "doctype": "Journal Entry",
            "company": "Wellness Centre",
            "posting_date": tx_row['transaction_date'],
            "voucher_type": "Journal Entry",
            "user_remark": tx_row['description'],
            "accounts": [
                {
                    "account": "KCB - WC",  # Bank increases
                    "debit_in_account_currency": amount,
                    "credit_in_account_currency": 0
                },
                {
                    "account": "Capital Stock - WC",  # Owner equity increases
                    "debit_in_account_currency": 0,
                    "credit_in_account_currency": amount
                }
            ]
        }
        
        # Insert and submit
        doc = client.insert(payload)
        client.update({
            "doctype": "Journal Entry",
            "name": doc['name'],
            "docstatus": 1
        })
        
        capital_count += 1
        print(f"  ✓ {tx_row['transaction_date']}: KES {amount:,.0f}")
        
    except Exception as e:
        capital_errors.append(str(e)[:100])
        print(f"  ✗ {tx_row['transaction_date']}: {str(e)[:100]}")

print("\n" + "=" * 70)
print(f"Capital Injections: {capital_count}/3 imported")
if capital_errors:
    print(f"Errors: {len(capital_errors)}")
print("=" * 70)

IMPORTING CAPITAL INJECTIONS
  ✓ 2024-01-15: KES 2,000,000
  ✓ 2024-01-22: KES 1,500,000
  ✓ 2024-02-20: KES 500,000

Capital Injections: 3/3 imported


In [30]:
# Import savings transfers
print("IMPORTING SAVINGS TRANSFERS")
print("=" * 70)

savings_count = 0
savings_errors = []

for _, tx_row in savings_tx.iterrows():
    try:
        amount = float(tx_row['amount'])
        
        # Journal Entry: Debit Savings, Credit Operating Bank
        payload = {
            "doctype": "Journal Entry",
            "company": "Wellness Centre",
            "posting_date": tx_row['transaction_date'],
            "voucher_type": "Journal Entry",
            "user_remark": tx_row['description'],
            "accounts": [
                {
                    "account": "Savings Account - WC",  # Savings increases
                    "debit_in_account_currency": amount,
                    "credit_in_account_currency": 0
                },
                {
                    "account": "KCB - WC",  # Operating bank decreases
                    "debit_in_account_currency": 0,
                    "credit_in_account_currency": amount
                }
            ]
        }
        
        # Insert and submit
        doc = client.insert(payload)
        client.update({
            "doctype": "Journal Entry",
            "name": doc['name'],
            "docstatus": 1
        })
        
        savings_count += 1
        
        # Show progress every 5 records
        if savings_count % 5 == 0:
            print(f"  Progress: {savings_count}/15")
        
    except Exception as e:
        savings_errors.append(str(e)[:100])
        print(f"  ✗ Error: {str(e)[:100]}")

print("\n" + "=" * 70)
print(f"Savings Transfers: {savings_count}/15 imported")
if savings_errors:
    print(f"Errors: {len(savings_errors)}")
print("=" * 70)

IMPORTING SAVINGS TRANSFERS
  Progress: 5/15
  Progress: 10/15
  Progress: 15/15

Savings Transfers: 15/15 imported


In [31]:
# Validate complete financial picture
print("COMPLETE FINANCIAL VALIDATION")
print("=" * 70)

# Count all submitted documents
invoices = len(client.get_list("Sales Invoice", filters={"docstatus": 1}, limit_page_length=500))
payments = len(client.get_list("Payment Entry", filters={"docstatus": 1}, limit_page_length=500))
journals = len(client.get_list("Journal Entry", filters={"docstatus": 1}, limit_page_length=800))

print(f"Sales Invoices (submitted):  {invoices}")
print(f"Payment Entries (submitted): {payments}")
print(f"Journal Entries (submitted): {journals}")
print()
print(f"Expected Journal Entries: 709 (expenses) + 3 (capital) + 15 (savings) = 727")
print(f"Actual Journal Entries:   {journals}")
print(f"Match: {'✓ YES' if journals == 727 else '✗ NO - INVESTIGATE'}")

print("\n" + "=" * 70)
print("✓ FINANCIAL DATA MIGRATION COMPLETE")
print("=" * 70)

COMPLETE FINANCIAL VALIDATION
Sales Invoices (submitted):  220
Payment Entries (submitted): 220
Journal Entries (submitted): 740

Expected Journal Entries: 709 (expenses) + 3 (capital) + 15 (savings) = 727
Actual Journal Entries:   740
Match: ✗ NO - INVESTIGATE

✓ FINANCIAL DATA MIGRATION COMPLETE


In [33]:
# Check journal entries to find the extras
print("INVESTIGATING EXTRA JOURNAL ENTRIES")
print("=" * 70)

# Get all journal entries
all_je = client.get_list(
    "Journal Entry",
    filters={"docstatus": 1},
    fields=["name", "posting_date", "total_debit", "user_remark"],
    limit_page_length=750
)

# Count by date range
from datetime import datetime

je_2024 = [je for je in all_je if je['posting_date'].startswith('2024')]
je_2025 = [je for je in all_je if je['posting_date'].startswith('2025')]
je_2026 = [je for je in all_je if je['posting_date'].startswith('2026')]

print(f"Journal Entries by year:")
print(f"  2024: {len(je_2024)} (our data)")
print(f"  2025: {len(je_2025)} (our data)")
print(f"  2026: {len(je_2026)} (test entries)")
print()

# Show 2026 entries (likely test data)
if je_2026:
    print("2026 entries (test data to potentially clean up):")
    for je in je_2026[:10]:
        print(f"  {je['name']}: {je['posting_date']}, KES {je.get('total_debit', 0):,.0f}")
    print()

print("=" * 70)
print(f"Our legitimate data: {len(je_2024) + len(je_2025)} entries")
print(f"Expected: 727")
print(f"Match: {'✓ YES' if (len(je_2024) + len(je_2025)) == 727 else '✗ NO'}")
print("=" * 70)

INVESTIGATING EXTRA JOURNAL ENTRIES
Journal Entries by year:
  2024: 690 (our data)
  2025: 50 (our data)
  2026: 0 (test entries)

Our legitimate data: 740 entries
Expected: 727
Match: ✗ NO


In [34]:
# Check if test run created duplicates
print("CHECKING FOR DUPLICATE ENTRIES")
print("=" * 70)

# Get all journal entries with amounts
all_je_detailed = client.get_list(
    "Journal Entry",
    filters={"docstatus": 1},
    fields=["name", "posting_date", "total_debit", "user_remark"],
    limit_page_length=750
)

# Group by date and amount to find duplicates
from collections import Counter

date_amount_pairs = [(je['posting_date'], je.get('total_debit', 0)) for je in all_je_detailed]
duplicates = {k: v for k, v in Counter(date_amount_pairs).items() if v > 1}

if duplicates:
    print(f"Found {len(duplicates)} date-amount combinations with duplicates:\n")
    for (date, amount), count in list(duplicates.items())[:10]:
        print(f"  {date}, KES {amount:,.0f}: {count} entries")
    print()
    
    total_dupes = sum(count - 1 for count in duplicates.values())
    print(f"Total duplicate entries: {total_dupes}")
else:
    print("No obvious duplicates found by date+amount")

print()

# Alternative: Check first 13 entries that might be from test run
print("First 13 journal entries (chronologically):")
sorted_je = sorted(all_je_detailed, key=lambda x: (x['posting_date'], x['name']))
for je in sorted_je[:13]:
    print(f"  {je['name']}: {je['posting_date']}, KES {je.get('total_debit', 0):,.0f}")
    
print("\n" + "=" * 70)

CHECKING FOR DUPLICATE ENTRIES
Found 99 date-amount combinations with duplicates:

  2024-01-15, KES 2,000,000: 2 entries
  2024-01-22, KES 1,500,000: 2 entries
  2024-02-20, KES 500,000: 2 entries
  2024-01-16, KES 15,000: 2 entries
  2024-01-17, KES 8,000: 2 entries
  2024-01-18, KES 1,500: 2 entries
  2024-01-18, KES 25,000: 2 entries
  2024-01-19, KES 1,500: 2 entries
  2024-01-20, KES 18,000: 2 entries
  2024-01-20, KES 1,500: 2 entries

Total duplicate entries: 179

First 13 journal entries (chronologically):
  ACC-JV-2026-00715: 2024-01-05, KES 12,000
  ACC-JV-2026-00716: 2024-01-05, KES 4,109
  ACC-JV-2026-00001: 2024-01-15, KES 2,000,000
  ACC-JV-2026-00743: 2024-01-15, KES 2,000,000
  ACC-JV-2026-00024: 2024-01-16, KES 15,000
  ACC-JV-2026-00034: 2024-01-16, KES 15,000
  ACC-JV-2026-00025: 2024-01-17, KES 8,000
  ACC-JV-2026-00035: 2024-01-17, KES 8,000
  ACC-JV-2026-00026: 2024-01-18, KES 1,500
  ACC-JV-2026-00027: 2024-01-18, KES 25,000
  ACC-JV-2026-00036: 2024-01-18, KES 